In [0]:
%sql
CREATE TABLE dev.mohit_gangwani.latest_airing_of_shows
SELECT fk_show_id, fk_station_id, airdate
FROM prod.detection.epg_schedule sch
WHERE 

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_filtered_set;
CREATE TABLE dev.mohit_gangwani.ad_labeling_filtered_set AS
SELECT vc.external_id AS ad_id
, NVL(vc.fk_dma_id, 0) AS fk_dma_id
FROM prod.detection.viewing_commercials_firehose_dedup vc
LEFT JOIN (
  SELECT cief.fk_commercial_id, cief.external_id
  FROM prod.detection.commercial_id_external_firehose cief
  JOIN prod.detection.clients cl
    ON cl.client_id = cief.fk_client_id
  WHERE cl.client_name = 'kinetiq'
  GROUP BY 1, 2
) cief
  ON cief.external_id = vc.external_id
WHERE vc.session_start >= CURRENT_DATE - 8
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
  AND (cief.external_id IS NOT NULL OR vc.fk_commercial_source_id = 2)
GROUP BY 1, 2
HAVING COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) >= 20
;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_base_viewing_table;
CREATE TABLE dev.mohit_gangwani.ad_labeling_base_viewing_table AS
SELECT vc.fk_tvid
, vc.session_start
, COALESCE(vc.external_id, cief.ad_id) AS ad_id
, CASE WHEN UPPER(COALESCE(vc.prev_content_type, content.content_type)) <=> 'LINEAR'
         OR UPPER(COALESCE(vc.reported_input_source, content.reported_input_source)) <=> 'ANTENNA'
         OR COALESCE(vc.prev_station_id, content.tuner_channel_id, content.tms_tuner_channel_id) IS NOT NULL THEN 'LINEAR'
       WHEN UPPER(COALESCE(vc.prev_content_type, content.content_type)) <=> 'STREAMING'
         OR COALESCE(vc.prev_vizio_epg_station, content.vizio_epg_station) IS NOT NULL
         OR UPPER(COALESCE(vc.reported_input_source, content.reported_input_source)) <=> 'APPS' THEN 'APPS'
       ELSE 'UNKNOWN' END AS app_or_linear
, CASE WHEN st.local_or_national = 'Local' OR COALESCE(content.tuner_channel_id, content.tms_tuner_channel_id) IS NOT NULL THEN 'Local'
       WHEN st.local_or_national IS NOT NULL THEN 'National'
       ELSE 'Unknown' END AS station_type
, CASE WHEN content.session_start <= TIMESTAMPADD(SECOND, content.runtime, content.airdate) THEN 'Live'
       WHEN COALESCE(content.tuner_channel_id, content.tms_tuner_channel_id) IS NOT NULL THEN 'Timeshifted'
       WHEN COALESCE(vc.prev_vizio_epg_station, content.vizio_epg_station) IS NOT NULL THEN 'Live'
       WHEN content.is_live = TRUE THEN 'Live'
       WHEN content.is_live = FALSE THEN 'Timeshifted'
  END AS is_live
, NVL(vc.fk_dma_id, 0) AS fk_dma_id
, DATE_TRUNC('day', vc.session_start) AS event_date
, DATE_TRUNC('HOUR', vc.session_start) AS time_bin_start
, DATE_TRUNC('HOUR', vc.session_start) + INTERVAL '1 HOUR' AS time_bin_end
-- , TO_TIMESTAMP(FLOOR(UNIX_TIMESTAMP(vc.session_start) / 600) * 600) AS time_bin_start
-- , TO_TIMESTAMP(FLOOR(UNIX_TIMESTAMP(vc.session_start) / 600) * 600 + 600) AS time_bin_end
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN dev.mohit_gangwani.ad_labeling_filtered_set cief
  ON cief.ad_id = vc.external_id
 AND cief.fk_dma_id = NVL(vc.fk_dma_id, 0)
LEFT JOIN prod.detection.viewing_content_firehose content
  ON vc.fk_tvid = content.fk_tvid
 AND vc.prev_session_start = content.session_start
 AND content.session_start >= CURRENT_DATE - 9
LEFT JOIN prod.detection.epg_station st
  ON st.station_id = COALESCE(vc.prev_station_id, content.fk_station_id)
 AND st.vendor_name = 'TIVO'
WHERE vc.session_start >= CURRENT_DATE - 8
  AND vc.session_start < CURRENT_DATE
  AND vc.fk_zoo_id = 17
GROUP BY ALL;

In [0]:
%sql
-- # Active TVs per DMA × 10-min bin × platform/station
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_opportunities_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_opportunities_1_hour_binned AS
SELECT time_bin_start
, fk_dma_id
, app_or_linear
, station_type
, COUNT(DISTINCT fk_tvid) AS active_tvs
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table
GROUP BY 1,2,3,4

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_impression_count_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_impression_count_1_hour_binned AS
SELECT n.time_bin_start
, n.fk_dma_id
, n.app_or_linear
, n.station_type
, n.ad_id
, COUNT(DISTINCT fk_tvid||'_'||session_start) AS impressions
, SUM(CASE WHEN n.is_live = 'Live' THEN 1 ELSE 0 END) AS live_impressions
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table n
GROUP BY 1,2,3,4,5;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_phat_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_phat_1_hour_binned AS
SELECT i.time_bin_start
, i.fk_dma_id
, i.app_or_linear
, i.station_type
, i.ad_id
, i.impressions
, COALESCE(o.active_tvs, 0) AS opportunities
, CASE WHEN COALESCE(o.active_tvs,0) > 0 THEN i.impressions / o.active_tvs
       ELSE 0.0
  END AS p_hat_dma
, CASE WHEN i.impressions > 0 THEN i.live_impressions / i.impressions END AS live_share
FROM dev.mohit_gangwani.ad_labeling_impression_count_1_hour_binned i
LEFT JOIN dev.mohit_gangwani.ad_labeling_opportunities_1_hour_binned o
  ON i.time_bin_start = o.time_bin_start
 AND i.fk_dma_id = o.fk_dma_id
 AND i.app_or_linear = o.app_or_linear
 AND i.station_type = o.station_type
GROUP BY ALL;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
ORDER BY impressions DESC
LIMIT 100

In [0]:
%sql
SELECT * FROM prod.detection.commercial_id_external_firehose WHERE external_id = 'AE16546-2025-43-00305'

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_coverage_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_coverage_1_hour_binned AS
WITH flags AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , ad_id
  , COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY 1, 2, 3, 4
)
, ttl AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY 1, 2, 3
)
SELECT a.time_bin_start
, a.app_or_linear
, a.station_type
, a.ad_id
, a.dma_count/t.dma_count AS coverage_score
FROM flags a
LEFT JOIN ttl t
  ON t.time_bin_start = a.time_bin_start
 AND t.app_or_linear = a.app_or_linear
 AND t.station_type = a.station_type
GROUP BY ALL;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_coverage_1_hour_binned
ORDER BY coverage_score
LIMIT 100

In [0]:
%sql
-- Entropy across DMAs in a bin
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_entropy_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_entropy_1_hour_binned AS
WITH base AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
)
, tot AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , ad_id
  , SUM(impressions)*1.0 AS tot_imp
  , COUNT(DISTINCT fk_dma_id) AS k_dmas
  FROM base
  GROUP BY 1,2,3,4
)
, joined AS (
  SELECT b.time_bin_start
  , b.app_or_linear
  , b.station_type
  , b.ad_id
  , b.fk_dma_id
  , b.impressions
  , t.tot_imp
  , t.k_dmas
  , CASE WHEN t.tot_imp>0 THEN b.impressions / t.tot_imp ELSE 0.0 END AS p_dma
  FROM base b
  JOIN tot t 
    ON b.time_bin_start = t.time_bin_start
   AND b.app_or_linear = t.app_or_linear
   AND b.station_type = t.station_type
   AND b.ad_id = t.ad_id
)
SELECT time_bin_start
, app_or_linear
, station_type
, ad_id
, -SUM(CASE WHEN p_dma>0 THEN p_dma * LOG(p_dma) ELSE 0 END) AS entropy
, MAX(k_dmas) AS k_dmas
, CASE WHEN MAX(k_dmas) > 1 THEN (-SUM(CASE WHEN p_dma>0 THEN p_dma * LOG(p_dma) ELSE 0 END)) / LOG(MAX(k_dmas)) ELSE 0.0 END AS entropy_norm
FROM joined
GROUP BY 1,2,3,4;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_entropy_1_hour_binned
WHERE k_dmas = 2
ORDER BY entropy_norm
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_dma_clt_baseline_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_dma_clt_baseline_1_hour_binned AS
WITH totals AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , fk_dma_id
  , SUM(impressions) AS imp_all
  , SUM(opportunities) AS opp_all
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY 1,2,3,4
)
, per_ad AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , fk_dma_id
  , ad_id
  , impressions AS imp_ad
  , opportunities AS opp_ad
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
)
SELECT a.time_bin_start
, a.app_or_linear
, a.station_type
, a.fk_dma_id
, a.ad_id
, GREATEST(t.imp_all - a.imp_ad, 0) AS imp_others
, GREATEST(t.opp_all - a.opp_ad, 0) AS opp_others
, CASE WHEN (t.opp_all - a.opp_ad) > 0 THEN (t.imp_all - a.imp_ad) / (t.opp_all - a.opp_ad)
       ELSE 0.0
  END AS p_baseline
FROM per_ad a
JOIN totals t
  ON a.time_bin_start = t.time_bin_start
 AND a.app_or_linear = t.app_or_linear
 AND a.station_type = t.station_type
 AND a.fk_dma_id = t.fk_dma_id;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_dma_clt_baseline_1_hour_binned
ORDER BY p_baseline
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_clt_ztest_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_clt_ztest_1_hour_binned AS
SELECT p.time_bin_start
, p.app_or_linear
, p.station_type
, p.fk_dma_id
, p.ad_id
, p.impressions
, p.opportunities
, p.p_hat_dma
, b.p_baseline
, b.opp_others
, SQRT(GREATEST(p.p_hat_dma * (1 - p.p_hat_dma) / NULLIF(p.opportunities, 0), 0.0) + GREATEST(b.p_baseline * (1 - b.p_baseline) / NULLIF(b.opp_others,0), 0.0)) AS se_diff
FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned p
JOIN dev.mohit_gangwani.ad_labeling_dma_clt_baseline_1_hour_binned b
  ON p.time_bin_start = b.time_bin_start
  AND p.app_or_linear = b.app_or_linear
  AND p.station_type = b.station_type
  AND p.fk_dma_id = b.fk_dma_id
  AND p.ad_id = b.ad_id;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_clt_ztest_1_hour_binned
ORDER BY se_diff DESC
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_pvals_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_pvals_1_hour_binned AS
WITH z_calc AS (
  SELECT *
  , CASE WHEN se_diff > 0 THEN (p_hat_dma - p_baseline) / se_diff
         ELSE 0
    END AS z
  FROM dev.mohit_gangwani.ad_labeling_clt_ztest_1_hour_binned
)
, phi_calc AS (
  SELECT *
  , ABS(z) AS abs_z
  , 1.0/(1.0 + 0.2316419 * ABS(z)) AS t
  , EXP(-0.5 * POWER(ABS(z), 2)) / SQRT(2.0 * PI()) AS phi_z
  FROM z_calc
)
, abs_phi_calc AS (
  SELECT *
  , CASE WHEN se_diff <= 0 THEN 0.5
        ELSE 1.0 - (phi_z * ((0.319381530 * t) - (0.356563782 * POWER(t, 2)) + (1.781477937 * POWER(t, 3)) - (1.821255978 * POWER(t, 4)) + (1.330274429 * POWER(t, 5))))
    END AS cdf_abs_z_appox
  FROM phi_calc
)
SELECT *
, CASE WHEN se_diff <= 0 THEN 1.0
       ELSE 2.0 * (1.0 - cdf_abs_z_appox)
  END AS p_value
FROM abs_phi_calc
;

In [0]:

%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_pvals_1_hour_binned
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_sig_coverage_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_sig_coverage_1_hour_binned AS
WITH ranked AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , fk_dma_id
  , ad_id
  , p_value
  , ROW_NUMBER() OVER (PARTITION BY time_bin_start, app_or_linear, station_type ORDER BY p_value) AS rank_k
  , COUNT(*) OVER (PARTITION BY time_bin_start, app_or_linear, station_type) AS m_tests
  FROM dev.mohit_gangwani.ad_labeling_pvals_1_hour_binned
)
SELECT *
, (rank_k * 1.0 / m_tests) * 0.05 AS bh_threshold
, CASE WHEN p_value <= (rank_k * 1.0  / m_tests) * 0.05 THEN 1 ELSE 0
  END AS reject_h0
FROM ranked;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_sig_coverage_1_hour_binned
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_sig_count_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_sig_count_1_hour_binned AS
SELECT time_bin_start
, app_or_linear
, station_type
, ad_id
, SUM(CASE WHEN reject_h0 = 1 THEN 1 ELSE 0 END) AS sig_dma_count_bh05
FROM dev.mohit_gangwani.ad_labeling_sig_coverage_1_hour_binned
GROUP BY 1,2,3,4;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_one;
CREATE TABLE dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_one AS
WITH ad_mass AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY ALL
)
SELECT time_bin_start
, app_or_linear
, station_type
, ad_id
, SUM(impressions) AS tot_ad
FROM ad_mass
GROUP BY 1,2,3,4

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_two;
CREATE TABLE dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_two AS
WITH mkt_mass AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , fk_dma_id
  , SUM(impressions) AS imp_mkt
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY 1,2,3,4
)
, mkt_tot AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , SUM(imp_mkt) AS tot_mkt
  FROM mkt_mass
  GROUP BY 1,2,3
)
SELECT m.time_bin_start
, m.app_or_linear
, m.station_type
, m.fk_dma_id
, m.imp_mkt
, mt.tot_mkt
FROM mkt_mass m
JOIN mkt_tot mt
  ON m.time_bin_start = mt.time_bin_start
 AND m.app_or_linear = mt.app_or_linear
 AND m.station_type = mt.station_type
GROUP BY ALL

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_three;
CREATE TABLE dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_three AS
SELECT a.time_bin_start
, a.app_or_linear
, a.station_type
, a.ad_id
, a.fk_dma_id
, a.impressions
, t.tot_ad
, m.imp_mkt
, m.tot_mkt
FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned a
JOIN dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_one t
  ON a.time_bin_start = t.time_bin_start
  AND a.app_or_linear = t.app_or_linear
  AND a.station_type = t.station_type
  AND a.ad_id = t.ad_id
JOIN dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_two m
  ON a.time_bin_start = m.time_bin_start
  AND a.app_or_linear = m.app_or_linear
  AND a.station_type = m.station_type
  AND a.fk_dma_id = m.fk_dma_id
GROUP BY ALL

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned AS
WITH den AS (
  SELECT DISTINCT time_bin_start
  , app_or_linear
  , station_type
  , ad_id
  , 1e-9 + impressions / NULLIF(tot_ad, 0) AS p_ad
  , 1e-9 + imp_mkt / NULLIF(tot_mkt, 0) AS q_mkt
  , LOG(p_ad / q_mkt) AS lpq
  , p_ad * lpq AS kl
  FROM dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned_part_three
)
SELECT time_bin_start
, app_or_linear
, station_type
, ad_id
, SUM(kl) AS kl_dma_vs_market
FROM den
GROUP BY 1,2,3,4;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned
ORDER BY kl_dma_vs_market
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_station_mix_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_station_mix_1_hour_binned AS
WITH ad_s AS (
  SELECT time_bin_start
  , app_or_linear
  , ad_id
  , station_type
  , SUM(impressions) AS imp
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY 1, 2, 3, 4
)
, ad_tot AS (
  SELECT time_bin_start
  , app_or_linear
  , ad_id
  , SUM(imp) AS tot
  FROM ad_s
  GROUP BY 1, 2, 3
)
, mkt_s AS (
  SELECT time_bin_start
  , app_or_linear
  , station_type
  , SUM(impressions) AS imp_mkt
  FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned
  GROUP BY 1, 2, 3
)
, mkt_tot AS (
  SELECT time_bin_start
  , app_or_linear
  , SUM(imp_mkt) AS tot_mkt
  FROM mkt_s
  GROUP BY 1, 2
)
SELECT a.time_bin_start
, a.app_or_linear
, a.ad_id
, a.station_type
, (a.imp / NULLIF(t.tot, 0)) AS share_ad, (m.imp_mkt / NULLIF(mt.tot_mkt, 0)) AS share_mkt
, (a.imp / NULLIF(t.tot, 0)) / NULLIF((m.imp_mkt / NULLIF(mt.tot_mkt, 0)), 0) AS mix_ratio_station
FROM ad_s a
JOIN ad_tot t
  ON a.time_bin_start = t.time_bin_start
 AND a.app_or_linear = t.app_or_linear
 AND a.ad_id = t.ad_id
JOIN mkt_s m
  ON a.time_bin_start = m.time_bin_start
 AND a.app_or_linear = m.app_or_linear
 AND a.station_type = m.station_type
JOIN mkt_tot mt
  ON a.time_bin_start = mt.time_bin_start
 AND a.app_or_linear = mt.app_or_linear;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_station_mix_1_hour_binned
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_cocoverage_concurrency_1_hour_binned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_cocoverage_concurrency_1_hour_binned AS
WITH per_bin_dma AS (
  SELECT time_bin_start
  , fk_dma_id
  , app_or_linear
  , station_type
  , COUNT(DISTINCT ad_id) AS ads_in_bin_dma
  FROM dev.mohit_gangwani.ad_labeling_impression_count_1_hour_binned
  GROUP BY 1,2,3,4
),
per_ad_bin_dma AS (
  SELECT i.time_bin_start
  , i.fk_dma_id
  , i.app_or_linear
  , i.station_type
  , i.ad_id
  , b.ads_in_bin_dma
  FROM dev.mohit_gangwani.ad_labeling_impression_count_1_hour_binned i
  JOIN per_bin_dma b
    ON i.time_bin_start = b.time_bin_start
   AND i.fk_dma_id = b.fk_dma_id
   AND i.app_or_linear = b.app_or_linear
   AND i.station_type = b.station_type
)
SELECT time_bin_start
, app_or_linear
, station_type
, ad_id
, AVG(ads_in_bin_dma - 1) AS competitive_pressure
FROM per_ad_bin_dma
GROUP BY 1,2,3,4;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_cocoverage_concurrency_1_hour_binned
ORDER BY competitive_pressure
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_final_table_new;
CREATE TABLE dev.mohit_gangwani.ad_labeling_final_table_new AS
SELECT p.time_bin_start
, DATE_TRUNC('DAY', p.time_bin_start) AS event_date
, p.app_or_linear
, p.station_type
, p.ad_id
 -- aggregate across DMAs in bin
, SUM(p.impressions) AS impressions
, SUM(p.opportunities) AS opportunities
, SUM(p.impressions) / NULLIF(SUM(p.opportunities), 0) AS p_hat_bin
, AVG(p.live_share) AS live_share
, c.coverage_score
, e.entropy_norm
, s.sig_dma_count_bh05
, kl.kl_dma_vs_market
, ms.mix_ratio_station
, cp.competitive_pressure
FROM dev.mohit_gangwani.ad_labeling_phat_1_hour_binned p
LEFT JOIN dev.mohit_gangwani.ad_labeling_coverage_1_hour_binned c
  ON p.time_bin_start = c.time_bin_start
 AND p.app_or_linear = c.app_or_linear
 AND p.station_type = c.station_type
 AND p.ad_id = c.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_entropy_1_hour_binned e
  ON p.time_bin_start = e.time_bin_start
 AND p.app_or_linear = e.app_or_linear
 AND p.station_type = e.station_type
 AND p.ad_id = e.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_sig_count_1_hour_binned s
  ON p.time_bin_start = s.time_bin_start
 AND p.app_or_linear = s.app_or_linear
 AND p.station_type = s.station_type
 AND p.ad_id = s.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_klvmarket_1_hour_binned kl
  ON p.time_bin_start = kl.time_bin_start
 AND p.app_or_linear = kl.app_or_linear
 AND p.station_type = kl.station_type
 AND p.ad_id = kl.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_station_mix_1_hour_binned ms
  ON p.time_bin_start = ms.time_bin_start
 AND p.app_or_linear = ms.app_or_linear
 AND p.ad_id=ms.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_cocoverage_concurrency_1_hour_binned cp
  ON p.time_bin_start = cp.time_bin_start
 AND p.app_or_linear = cp.app_or_linear
 AND p.station_type = cp.station_type
 AND p.ad_id = cp.ad_id
GROUP BY p.time_bin_start, p.app_or_linear, p.station_type, p.ad_id, c.coverage_score, e.entropy_norm, s.sig_dma_count_bh05, kl.kl_dma_vs_market, ms.mix_ratio_station, cp.competitive_pressure;

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.ad_labeling_final_table_new
ORDER BY impressions
LIMIT 100

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_ad_level_final_table_new;
CREATE TABLE dev.mohit_gangwani.ad_labeling_ad_level_final_table_new AS
SELECT ad_id
, COUNT(*) AS num_rows -- number of (bin × slice) rows
, COUNT(DISTINCT time_bin_start) AS num_bins -- number of 10-min bins
, SUM(impressions) AS total_impressions
, SUM(opportunities) AS total_opportunities
-- pooled p-hat: sum(impressions) / sum(opportunities)
, CASE WHEN SUM(opportunities) > 0 THEN SUM(impressions) * 1.0 / SUM(opportunities)
       ELSE 0.0 
  END AS p_hat_pooled
-- Impression-weighted averages (bins with more delivery count more)
-- coverage: how many DMAs the ad tends to reach, weighted by delivery
, CASE WHEN SUM(impressions) > 0 THEN SUM(coverage_score * impressions) / SUM(impressions)
  END AS avg_coverage_weighted
-- entropy: how evenly impressions are spread across DMAs
, CASE WHEN SUM(impressions) > 0 THEN SUM(entropy_norm * impressions) / SUM(impressions)
  END AS avg_entropy_weighted
-- KL vs market: how different the DMA pattern is from baseline
, CASE WHEN SUM(impressions) > 0 THEN SUM(kl_dma_vs_market * impressions) / SUM(impressions)
  END AS avg_kl_weighted
-- live share: weighted by impressions (bins with more delivery matter more)
, CASE WHEN SUM(CASE WHEN live_share IS NOT NULL THEN impressions ELSE 0 END) > 0
            THEN SUM(COALESCE(live_share,0.0) * impressions)/ SUM(CASE WHEN live_share IS NOT NULL THEN impressions ELSE 0 END)
  END AS avg_live_share_weighted
-- station mix ratio: how much the ad over/under-indexes stations, weighted by delivery
, CASE WHEN SUM(CASE WHEN mix_ratio_station IS NOT NULL THEN impressions ELSE 0 END) > 0
            THEN SUM(COALESCE(mix_ratio_station, 0.0) * impressions) / SUM(CASE WHEN mix_ratio_station IS NOT NULL THEN impressions ELSE 0 END)
  END AS avg_mix_ratio_station_weighted
-- sig_dma_count: typical vs peak behavior
-- impression-weighted average sig DMA count (typical behavior)
, CASE WHEN SUM(impressions) > 0 THEN SUM(sig_dma_count_bh05 * impressions) / SUM(impressions)
  END AS avg_sig_dma_weighted
-- max sig DMA count (peak overdelivery moment)
, MAX(sig_dma_count_bh05) AS max_sig_dma
-- competitive pressure: environment-level, simple averages are fine
, AVG(competitive_pressure) AS avg_competitive_pressure
, MIN(competitive_pressure) AS min_competitive_pressure
, MAX(competitive_pressure) AS max_competitive_pressure
FROM dev.mohit_gangwani.ad_labeling_final_table_new
GROUP BY 1;

In [0]:
%sql
SELECT COUNT(*) FROM dev.mohit_gangwani.ad_labeling_ad_level_final_table_new